<a href="https://colab.research.google.com/github/srilakshmi005/careai-healthcare-ai/blob/main/CareAI_ML_Risk_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CareAI: ML Risk Prediction Model
## 30-Day Hospital Readmission Risk Prediction

Production-ready machine learning pipeline for predicting 30-day hospital readmission risk using XGBoost, with comprehensive model comparison, data leakage detection, and threshold optimization.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import joblib
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Verify library versions
print("CareAI project started successfully!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Scikit-learn: {pd.__version__}")
print("All ML libraries are ready! ✅")

## Step 1: Load Dataset

Load diabetes/readmission dataset from UCI ML Repository

In [ ]:
from ucimlrepo import fetch_ucirepo

# Load dataset from UCI
diabetes = fetch_ucirepo(id=296)
X = diabetes.data.features
y = diabetes.data.targets

print("Dataset loaded successfully! ✅")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print("\n===== DATASET INFORMATION =====")
print(f"Rows: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print("\n===== FIRST 5 ROWS =====")
print(X.head())

## Step 2: Exploratory Data Analysis

In [ ]:
# Analyze target variable and missing data
print("\n===== TARGET VALUES =====")
print(y.head())
print("\n===== MISSING VALUES =====")
print(X.isnull().sum().sort_values(ascending=False).head(10))

# Create binary target: readmitted within 30 days
y_binary = (y['readmitted'] == '<30').astype(int)
print("\n===== BINARY TARGET =====")
print(y_binary.value_counts())
print("\n0 = Not readmitted within 30 days")
print("1 = Readmitted within 30 days")

# Missing data report
missing_report = pd.DataFrame({
    'missing_count': X.isnull().sum(),
    'missing_percent': (X.isnull().sum() / len(X) * 100).round(2)
}).sort_values('missing_count', ascending=False)
missing_report = missing_report[missing_report['missing_count'] > 0]
print("\n===== MISSING DATA REPORT =====")
print(missing_report)

## Step 3: Data Preprocessing & Feature Engineering

In [ ]:
# Handle missing values
X_clean = X.copy()

print(f"Original features: {X_clean.shape[1]}")
print(f"Features after removing identifiers: {X_clean.shape[1]}")

# Separate numeric and categorical features
numeric_features = X_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_clean.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print("\n===== TRAIN / TEST SPLIT =====")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Training positive rate: {y_train.mean()*100:.2f} %")
print(f"Testing positive rate: {y_test.mean()*100:.2f} %")

## Step 4: Create Preprocessing Pipeline

In [ ]:
# Create preprocessing pipeline with standardization and one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

print("Preprocessing pipeline created successfully! ✅")

## Step 5: Baseline Model - Logistic Regression

In [ ]:
# Train Logistic Regression baseline
print("Training Logistic Regression...")
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_pred_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]
roc_auc_lr = roc_auc_score(y_test, y_pred_proba_lr)

print("Model training completed successfully! ✅")
print("\n===== LOGISTIC REGRESSION EVALUATION =====")
print(classification_report(y_test, y_pred_lr))
print(f"ROC-AUC: {roc_auc_lr:.3f}")

## Step 6: Random Forest Model

In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_pred_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print("Random Forest training completed! ✅")
print("\n===== RANDOM FOREST EVALUATION =====")
print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC: {roc_auc_rf:.4f}")

## Step 7: XGBoost Model (with potential leakage)

In [ ]:
# Train XGBoost
print("Training XGBoost...")
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'))
])
xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)
y_pred_proba_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

print("XGBoost training completed! ✅")
print("\n===== XGBOOST EVALUATION =====")
print(classification_report(y_test, y_pred_xgb))
print(f"ROC-AUC: {roc_auc_xgb:.3f}")

## Step 8: Threshold Optimization

In [ ]:
# Analyze different decision thresholds for F1 optimization
thresholds = np.arange(0.1, 0.55, 0.05)
print("===== THRESHOLD ANALYSIS =====")
best_f1 = 0
best_threshold = 0.5

for threshold in thresholds:
    y_pred_threshold = (y_pred_proba_xgb >= threshold).astype(int)
    precision = (y_pred_threshold & y_test).sum() / max(y_pred_threshold.sum(), 1)
    recall = (y_pred_threshold & y_test).sum() / y_test.sum()
    f1 = 2 * (precision * recall) / max(precision + recall, 1e-6)
    print(f"Threshold: {threshold:.2f} | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

## Step 9: Apply Optimal Threshold

In [ ]:
# Apply optimized threshold
y_pred_final = (y_pred_proba_xgb >= best_threshold).astype(int)
roc_auc_final = roc_auc_score(y_test, y_pred_proba_xgb)

print("===== FINAL MODEL EVALUATION =====")
print(f"Selected threshold: {best_threshold}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_final, 
                          target_names=['Not readmitted', 'Readmitted within 30 days']))
print(f"ROC-AUC: {roc_auc_final:.3f}")

## Step 10: Feature Importance Analysis

In [ ]:
# Extract and analyze feature importance
print("Calculating feature importance...")
xgb_model = xgb_pipeline.named_steps['classifier']
feature_importance = xgb_model.feature_importances_

# Get preprocessed feature names
preprocessor = xgb_pipeline.named_steps['preprocessor']
feature_names = []

# Add numeric feature names
feature_names.extend(numeric_features)

# Add categorical feature names
cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
feature_names.extend(cat_feature_names)

# Create importance dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\n===== TOP 15 FEATURES =====")
print(importance_df.head(15).to_string(index=False))

## Step 11: Remove Data Leakage Feature

**Important**: `discharge_disposition_id` is only known AFTER discharge, so it cannot be used for predictions at admission time. This is data leakage and must be removed for production.

In [ ]:
# Identify and remove leakage feature
leakage_feature = 'discharge_disposition_id'

X_production = X_clean.drop(columns=[leakage_feature])
print(f"Original features: {X_clean.shape[1]}")
print(f"Production features: {X_production.shape[1]}")
print(f"Leakage feature removed: {leakage_feature}")

# Re-split data for production model
X_train_prod, X_test_prod, y_train_prod, y_test_prod = train_test_split(
    X_production, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print("\n===== PRODUCTION TRAIN / TEST SPLIT =====")
print(f"Training samples: {X_train_prod.shape[0]}")
print(f"Testing samples: {X_test_prod.shape[0]}")
print(f"Training positive rate: {y_train_prod.mean()*100:.2f} %")
print(f"Testing positive rate: {y_test_prod.mean()*100:.2f} %")

## Step 12: Production Model Training

In [ ]:
# Create production preprocessing pipeline
numeric_features_prod = X_production.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features_prod = X_production.select_dtypes(include=['object']).columns.tolist()

preprocessor_prod = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_prod),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_prod)
    ]
)

print("Production preprocessing pipeline created! ✅")

# Train production XGBoost model
print("Training production XGBoost model...")
xgb_prod_pipeline = Pipeline([
    ('preprocessor', preprocessor_prod),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'))
])
xgb_prod_pipeline.fit(X_train_prod, y_train_prod)
y_pred_proba_prod = xgb_prod_pipeline.predict_proba(X_test_prod)[:, 1]
roc_auc_prod = roc_auc_score(y_test_prod, y_pred_proba_prod)

print("Production XGBoost training completed! ✅")
print("\n===== PRODUCTION MODEL =====")
print(f"ROC-AUC: {roc_auc_prod:.4f}")

## Step 13: Production Model Threshold Analysis

In [ ]:
# Threshold analysis for production model
print("===== PRODUCTION THRESHOLD ANALYSIS =====")
best_f1_prod = 0
best_threshold_prod = 0.5

for threshold in thresholds:
    y_pred_threshold = (y_pred_proba_prod >= threshold).astype(int)
    precision = (y_pred_threshold & y_test_prod).sum() / max(y_pred_threshold.sum(), 1)
    recall = (y_pred_threshold & y_test_prod).sum() / y_test_prod.sum()
    f1 = 2 * (precision * recall) / max(precision + recall, 1e-6)
    print(f"Threshold: {threshold:.2f} | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
    if f1 > best_f1_prod:
        best_f1_prod = f1
        best_threshold_prod = threshold

## Step 14: Model Comparison

In [ ]:
# Compare all models
print("===== CAREAI MODEL COMPARISON =====")
model_results = pd.DataFrame({
    'model': [
        'Logistic Regression',
        'Random Forest',
        'XGBoost (potential leakage)',
        'XGBoost (production)'
    ],
    'roc_auc': [
        roc_auc_lr,
        roc_auc_rf,
        roc_auc_xgb,
        roc_auc_prod
    ],
    'notes': [
        'Baseline model',
        'Improved baseline',
        'Included potentially unavailable discharge features',
        'Leakage feature removed'
    ]
})
print(model_results.to_string(index=False))

## Step 15: Save Production Model

In [ ]:
# Save the production model
model_version = 'careai_xgboost_v1'
model_file = 'careai_model_v1.joblib'
joblib.dump(xgb_prod_pipeline, model_file)

print("===== MODEL SAVED =====")
print(f"Model version: {model_version}")
print(f"File: {model_file}")
print(f"Threshold: {best_threshold_prod}")
print(f"ROC-AUC: {roc_auc_prod:.4f}")
print("Saved successfully! ✅")

# Save model metadata
model_metadata = {
    'project': 'CareAI',
    'model_version': model_version,
    'algorithm': 'XGBoost',
    'prediction_target': '30-day readmission',
    'production_features': X_production.shape[1],
    'training_rows': X_train_prod.shape[0],
    'testing_rows': X_test_prod.shape[0],
    'roc_auc': round(roc_auc_prod, 4),
    'decision_threshold': best_threshold_prod,
    'leakage_feature_removed': leakage_feature
}

print("\n===== CAREAI MODEL METADATA =====")
for key, value in model_metadata.items():
    print(f"{key} : {value}")

## Step 16: Store Data in SQLite

In [ ]:
# Create SQLite database for persistence
db_file = 'careai_healthcare.db'
conn = sqlite3.connect(db_file)

# Prepare data for storage
df_sql = pd.DataFrame({
    'patient_id': range(len(y_binary)),
    'readmitted_30d': y_binary.values
})

# Store in database
df_sql.to_sql('healthcare', conn, if_exists='replace', index=False)

print("CareAI SQLite database created successfully! ✅")
print(f"Rows stored in SQL: {len(df_sql)}")

# Query summary
sql_result = pd.read_sql('SELECT readmitted_30d, COUNT(*) as patient_count FROM healthcare GROUP BY readmitted_30d', conn)
print("\n===== SQL READMISSION SUMMARY =====")
print(sql_result.to_string(index=False))

conn.close()

## Summary

### Production-Ready ML Pipeline Completed ✅

#### Key Results:
- **Best Model**: XGBoost (Production) with ROC-AUC: 0.6606
- **Decision Threshold**: 0.15 (optimized for recall)
- **Top Features**: number_inpatient, diag_1, time_in_hospital
- **Data Quality**: Removed leakage feature (discharge_disposition_id)

#### Model Artifacts:
- `careai_model_v1.joblib` — Production-ready model
- `careai_healthcare.db` — SQLite database with patient data
- Model metadata for deployment and monitoring

#### Next Steps:
1. Deploy model to production API
2. Set up monitoring and retraining pipeline
3. Implement RAG system for clinical context
4. Integrate LLM evaluation framework
5. Monitor model drift and performance degradation